# Stage 2 - DenseNet-121 v2: Focal Loss (Softened Weights)

## 1. Giris ve Problem Tanimi

Bu notebook, DenseNet-121 model ailesinin **v2** iterasyonudur.
**Birincil metrik: Macro F1**

### Onceki Sonuclar

| Metrik | densenet_baseline | efnb3_v3 (best efnb3) |
|--------|------------------|------------------------|
| Macro F1 | **0.8494** | 0.8479 |
| ACA P / R / F1 | 0.7541 / 0.8214 / 0.7863 | 0.7458 / 0.7857 / 0.7652 |
| MCA P / R / F1 | 0.9710 / 0.8859 / 0.9265 | 0.9495 / 0.9185 / 0.9337 |
| PCA P / R / F1 | 0.7486 / 0.9448 / 0.8354 | 0.8101 / 0.8828 / 0.8449 |
| Test Acc | 89.31% | 90.03% |

### Baseline Analizi
- Baseline **Weighted CrossEntropyLoss** kullanarak tam ters orantili sinif agirliklari uygulamisti (ACA=4.13, MCA=0.47, PCA=1.59).
- Bu agresif agirliklar ACA recall'u yukseltti (0.8214) ama ACA precision dusuk kaldi (0.7541).
- PCA'da da benzer durum: recall cok yuksek (0.9448), precision dusuk (0.7486).
- **Sorun**: Tam ters orantili agirliklar precision-recall dengesini bozuyor.

### v2 Stratejisi: Focal Loss + Power-Law Softened Weights

EfficientNet-B3 deneylerinden ogrenilenler:
- **Focal Loss (gamma=1.5, power=0.5)** en iyi Macro F1 dengesini verdi (efnb3_v3: 0.8479)
- Power-law yumusatma, ACA-MCA arasindaki agirlik ucurumunu kapatir
- gamma=1.5 modelin zor orneklere odaklanmasini saglar

**Tek degisiklik**: Weighted CrossEntropyLoss → Focal Loss (gamma=1.5, power=0.5)
Diger her sey baseline ile **birebir ayni** (IMG_SIZE, LR, augmentation, scheduler, early stopping).

### Beklenen Etki
- ACA ve PCA precision artisi (agirliklar yumusatildigi icin)
- ACA recall hafif dusebilir (0.82 → 0.78-0.80) ama F1 dengesi iyilesecek
- **Hedef: Macro F1 > 0.86**

## Bolum 1: Paket Kurulumu

In [ ]:
import subprocess, sys

packages = [
    'torch', 'torchvision', 'torchaudio',
    'albumentations', 'opencv-python-headless',
    'scikit-learn', 'tqdm', 'seaborn'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)
print('Paketler yuklendi.')

## Bolum 2: Setup & Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import models

# Albumentations
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

# Metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from tqdm import tqdm
from collections import Counter

# Seed Setting
SEED = 42
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(SEED)

# Device Config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Bolum 3: Sabitler ve Hiperparametreler

In [ ]:
# --- PATHS ---
GCS_DATA_PATH = 'gs://stroke-detection/data/stroke_dataset/stroke_dataset/'

# --- CLASS INFO ---
CLASS_NAMES = ['ACA', 'MCA', 'PCA']
NUM_CLASSES = len(CLASS_NAMES)

# --- HYPERPARAMETERS (Baseline ile ayni) ---
IMG_SIZE = 300
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 100

# --- v2 DEGISIKLIK: Focal Loss Parametreleri ---
GAMMA = 1.5          # Focal Loss gamma (zor orneklere odaklanma)
WEIGHT_POWER = 0.5   # Sinif agirlik yumusatma (1.0=tam ters, 0.5=karekoku, 0.0=esit)

# Scheduler (Baseline ile ayni)
T_0 = 10
T_MULT = 2

# Early Stopping (Baseline ile ayni)
PATIENCE = 15

# Split Ratios (Baseline ile ayni)
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

print(f'=== DenseNet-121 v2 Parametreleri ===')
print(f'Model: DenseNet-121 (ImageNet pretrained)')
print(f'Loss: Focal Loss (gamma={GAMMA}, weight_power={WEIGHT_POWER})')
print(f'Learning Rate: {LEARNING_RATE}')
print(f'IMG_SIZE: {IMG_SIZE}, Batch: {BATCH_SIZE}')
print(f'Early Stopping: patience={PATIENCE} (val_macro_f1)')
print(f'Scheduler: CosineAnnealingWarmRestarts (T_0={T_0}, T_mult={T_MULT})')

## Bolum 4: Veri Erisimi (Ortam Tespiti)

In [ ]:
import os, subprocess
from pathlib import Path

if os.path.exists('/kaggle/input/stroke-images/flattened_images'):
    STROKE_IMAGES_DIR = '/kaggle/input/stroke-images/flattened_images'
    print('Ortam: Kaggle')
elif os.path.exists('/tmp/data/stroke_dataset/stroke_dataset') and len(os.listdir('/tmp/data/stroke_dataset/stroke_dataset')) >= 3:
    STROKE_IMAGES_DIR = '/tmp/data/stroke_dataset/stroke_dataset'
    print('Ortam: Vertex AI — Veri zaten mevcut')
else:
    print(f'GCS\'den veri indiriliyor: {GCS_DATA_PATH}')
    os.makedirs('/tmp/data/stroke_dataset', exist_ok=True)
    result = subprocess.run(
        ['gsutil', '-m', 'cp', '-r', GCS_DATA_PATH, '/tmp/data/stroke_dataset/'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'gsutil stderr: {result.stderr}')
        raise RuntimeError(f'GCS download failed: {result.stderr}')
    STROKE_IMAGES_DIR = '/tmp/data/stroke_dataset/stroke_dataset'
    if not os.path.exists(STROKE_IMAGES_DIR) or len(os.listdir(STROKE_IMAGES_DIR)) < 3:
        alt_path = '/tmp/data/stroke_dataset'
        if os.path.exists(os.path.join(alt_path, 'ACA')):
            STROKE_IMAGES_DIR = alt_path
            print(f'Veri alternatif dizinde bulundu: {alt_path}')
        else:
            for root, dirs, files in os.walk('/tmp/data'):
                print(f'  {root}: dirs={dirs}, files_count={len(files)}')
            raise RuntimeError(f'Veri indirme sonrasi klasor yapisi beklenenden farkli!')

print(f'STROKE_IMAGES_DIR: {STROKE_IMAGES_DIR}')
for cls in ['ACA', 'MCA', 'PCA']:
    cls_path = Path(STROKE_IMAGES_DIR, cls)
    count = len(list(cls_path.glob('*'))) if cls_path.exists() else 0
    print(f'  {cls}: {count} goruntu')
    if count == 0:
        raise RuntimeError(f'{cls} sinifi icin goruntu bulunamadi! Path: {cls_path}')

## Bolum 5: Veri Hazirligi (Stratified Split)

In [ ]:
def collect_stroke_image_paths(stroke_dir, class_names):
    image_paths = []
    labels = []
    for idx, class_name in enumerate(class_names):
        class_dir = Path(stroke_dir) / class_name
        if not class_dir.exists(): continue
        
        extensions = ['*.png', '*.jpg', '*.jpeg']
        class_images = []
        for ext in extensions:
            class_images.extend(list(class_dir.glob(ext)))
            
        for img_path in class_images:
            image_paths.append(str(img_path))
            labels.append(idx)
    return np.array(image_paths), np.array(labels)

all_image_paths, all_labels = collect_stroke_image_paths(STROKE_IMAGES_DIR, CLASS_NAMES)

# Stratified Split (70/15/15)
X_temp, X_test, y_temp, y_test = train_test_split(
    all_image_paths, all_labels, test_size=TEST_RATIO, stratify=all_labels, random_state=SEED
)
val_ratio_adjusted = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_ratio_adjusted, stratify=y_temp, random_state=SEED
)

print(f'Toplam: {len(all_image_paths)}')
print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')
print(f'Train sinif dagilimi: {dict(sorted(Counter(y_train).items()))}')
print(f'Val sinif dagilimi:   {dict(sorted(Counter(y_val).items()))}')
print(f'Test sinif dagilimi:  {dict(sorted(Counter(y_test).items()))}')

## Bolum 6: Augmentation & Dataset

Baseline ile birebir ayni augmentation pipeline.

**YASAKLAR (Medikal Guvenlik):**
- VerticalFlip: Beyin yukari-asagi simetrik degil
- GaussianBlur: DWI'da lezyon sinirlarini eritir
- CutMix: Kucuk ACA lezyonlarini kapatabilir

In [ ]:
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    
    # === GEOMETRIC (Safe for brain MRI) ===
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=0, p=0.5,
                       border_mode=cv2.BORDER_CONSTANT, value=0),
    
    # === ELASTIC DEFORMATION (Anatomic variation) ===
    A.ElasticTransform(alpha=50, sigma=50 * 0.05, p=0.3),
    A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.3),
    
    # === INTENSITY (Careful limits) ===
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
    A.GaussNoise(var_limit=(5.0, 20.0), p=0.2),
    
    # === NORMALIZATION ===
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

class StrokeDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = list(image_paths)
        self.labels = list(labels)
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = int(self.labels[idx])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
        return image, label

train_dataset = StrokeDataset(X_train, y_train, transform=train_transform)
val_dataset = StrokeDataset(X_val, y_val, transform=val_transform)
test_dataset = StrokeDataset(X_test, y_test, transform=val_transform)
print(f'Datasets: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}')

## Bolum 7: WeightedRandomSampler & Softened Class Weights (v2 OZEL)

**Baseline farki**: Sinif agirliklari `power=0.5` ile yumusatiliyor.

| | Baseline (power=1.0) | v2 (power=0.5) |
|---|---|---|
| ACA | 4.13 | ~1.53 |
| MCA | 0.47 | ~0.52 |
| PCA | 1.59 | ~0.95 |
| ACA/MCA orani | 8.79x | 2.94x |

In [ ]:
# 1. Sampler — Batch icinde sinif dengeleme (Baseline ile ayni)
class_counts_train = Counter(y_train)
class_weights_sampler = {c: 1.0 / count for c, count in class_counts_train.items()}
sample_weights = [class_weights_sampler[int(label)] for label in y_train]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=4, pin_memory=True)

# 2. Focal Loss icin Yumusatilmis Sinif Agirliklari (v2 DEGISIKLIK)
# Formul: weight = (total / (num_classes * count)) ^ WEIGHT_POWER
total_samples = sum(class_counts_train.values())
class_weights_loss = []

print('=== Sinif Agirlik Karsilastirmasi ===')
print(f'{"Sinif":<6} {"n":<6} {"Raw (p=1.0)":<14} {"Softened (p=0.5)":<18}')
print('-' * 45)

for i in range(NUM_CLASSES):
    count = class_counts_train[i]
    raw_weight = total_samples / (NUM_CLASSES * count)
    soft_weight = raw_weight ** WEIGHT_POWER
    class_weights_loss.append(soft_weight)
    print(f'{CLASS_NAMES[i]:<6} {count:<6} {raw_weight:<14.4f} {soft_weight:<18.4f}')

# Normalizasyon: toplam = NUM_CLASSES
weight_sum = sum(class_weights_loss)
class_weights_loss = [w * NUM_CLASSES / weight_sum for w in class_weights_loss]

print(f'\nNormalize Edilmis Final Agirliklar:')
for i, w in enumerate(class_weights_loss):
    print(f'  {CLASS_NAMES[i]}: {w:.4f}')

class_weights_tensor = torch.FloatTensor(class_weights_loss).to(device)

## Bolum 8: Focal Loss Tanimi (v2 OZEL)

Focal Loss formulu: `FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)`

- **gamma=1.5**: Kolay orneklerin loss katkisini azaltir, modeli zor orneklere odaklar
- **alpha (softened weights)**: Sinif dengesizligini telafi eder, ama yumusak sekilde

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

print(f'Focal Loss tanimlandi (gamma={GAMMA}, weight_power={WEIGHT_POWER})')

## Bolum 9: DenseNet-121 Model, Loss, Optimizer, Scheduler

In [ ]:
def create_model(num_classes):
    model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    num_features = model.classifier.in_features  # 1024
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(num_features, num_classes)
    )
    return model

model = create_model(NUM_CLASSES).to(device)

# v2 DEGISIKLIK: Focal Loss (baseline: Weighted CrossEntropyLoss)
criterion = FocalLoss(alpha=class_weights_tensor, gamma=GAMMA)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=T_0,
    T_mult=T_MULT,
    eta_min=1e-6
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: DenseNet-121')
print(f'Feature boyutu: {model.classifier[1].in_features}')
print(f'Total params: {total_params:,} | Trainable: {trainable_params:,}')
print(f'Loss: FocalLoss (gamma={GAMMA}, power={WEIGHT_POWER})')
print(f'Optimizer: AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})')
print(f'Scheduler: CosineAnnealingWarmRestarts (T_0={T_0}, T_mult={T_MULT})')

## Bolum 10: Egitim Fonksiyonlari

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc='Training', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
    return running_loss / total, correct / total


def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
    avg_loss = running_loss / total
    accuracy = correct / total
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    
    return avg_loss, accuracy, macro_f1, np.array(all_preds), np.array(all_labels), np.array(all_probs)

## Bolum 11: Egitim Dongusu + Early Stopping (Macro F1)

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
best_val_f1 = 0.0
patience_counter = 0
CHECKPOINT_PATH = 'best_model_densenet_v2.pth'

print('DenseNet-121 v2 Egitimi Basliyor...')
print(f'Loss: Focal Loss (gamma={GAMMA}, power={WEIGHT_POWER})')
print(f'LR={LEARNING_RATE}, Early Stopping: patience={PATIENCE}')
print('=' * 70)

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_f1, _, _, _ = validate(model, val_loader, criterion, device)
    
    scheduler.step(epoch)
    current_lr = optimizer.param_groups[0]['lr']
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    
    print(f'Epoch {epoch+1}/{NUM_EPOCHS} | LR: {current_lr:.2e}')
    print(f'  Train Loss: {train_loss:.4f} | Acc: {train_acc*100:.2f}%')
    print(f'  Val   Loss: {val_loss:.4f} | Acc: {val_acc*100:.2f}% | F1: {val_f1:.4f}')
    
    if val_f1 > best_val_f1:
        print(f'  >>> Iyilesme! (F1: {best_val_f1:.4f} -> {val_f1:.4f}). Model kaydediliyor...')
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), CHECKPOINT_PATH)
    else:
        patience_counter += 1
        print(f'  Early stopping counter: {patience_counter}/{PATIENCE}')
    
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping! {PATIENCE} epoch boyunca F1 iyilesmedi.')
        break
    
    print('-' * 70)

model.load_state_dict(torch.load(CHECKPOINT_PATH))
print(f'\nEgitim tamamlandi. Toplam {len(history["train_loss"])} epoch.')
print(f'En iyi model (Val F1: {best_val_f1:.4f}) yuklendi.')

## Bolum 12: Egitim Grafikleri

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', color='#2196F3')
axes[0].plot(history['val_loss'], label='Val Loss', color='#F44336')
axes[0].set_title('Training vs Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot([a*100 for a in history['train_acc']], label='Train Acc', color='#2196F3')
axes[1].plot([a*100 for a in history['val_acc']], label='Val Acc', color='#F44336')
axes[1].set_title('Training vs Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Macro F1
axes[2].plot(history['val_f1'], label='Val Macro F1', color='#4CAF50', linewidth=2)
axes[2].axhline(y=0.8494, color='gray', linestyle='--', alpha=0.7, label='Baseline F1 (0.8494)')
axes[2].set_title('Validation Macro F1')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Macro F1')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('DenseNet-121 v2 — Focal Loss (gamma=1.5, power=0.5)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Bolum 13: Test Degerlendirmesi — Classification Report

In [ ]:
print('=' * 70)
print('DenseNet-121 v2 — TEST SETI DEGERLENDIRMESI')
print('=' * 70)

test_loss, test_acc, test_f1, test_preds, test_labels, test_probs = validate(
    model, test_loader, criterion, device
)

print(f'\nTest Loss:     {test_loss:.4f}')
print(f'Test Accuracy: {test_acc*100:.2f}%')
print(f'Test Macro F1: {test_f1:.4f}')

print('\n' + '=' * 70)
print('Classification Report:')
print('=' * 70)
print(classification_report(
    test_labels, test_preds,
    target_names=CLASS_NAMES,
    digits=4
))

## Bolum 14: Confusion Matrix (Raw + Normalized)

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            annot_kws={'size': 14}, ax=axes[0])
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('True', fontsize=12)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')

# Normalized (per-class recall)
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            annot_kws={'size': 14}, ax=axes[1])
axes[1].set_xlabel('Predicted', fontsize=12)
axes[1].set_ylabel('True', fontsize=12)
axes[1].set_title('Normalized Confusion Matrix (Recall)', fontsize=13, fontweight='bold')

plt.suptitle('DenseNet-121 v2 — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Hata analizi
print('\n=== Hata Analizi ===')
for i, cls in enumerate(CLASS_NAMES):
    total_cls = cm[i].sum()
    correct_cls = cm[i, i]
    print(f'{cls}: {correct_cls}/{total_cls} dogru ({cm_norm[i, i]*100:.1f}%)', end='')
    errors = {CLASS_NAMES[j]: cm[i, j] for j in range(NUM_CLASSES) if j != i and cm[i, j] > 0}
    if errors:
        err_str = ', '.join([f'{k}={v}' for k, v in sorted(errors.items(), key=lambda x: -x[1])])
        print(f'  | Hatalar: {err_str}')
    else:
        print()

## Bolum 15: ROC Curves & AUC

In [ ]:
y_true_bin = label_binarize(test_labels, classes=list(range(NUM_CLASSES)))

plt.figure(figsize=(10, 8))

colors = ['#e74c3c', '#3498db', '#2ecc71']
auc_scores = []
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], test_probs[:, i])
    roc_auc = auc(fpr, tpr)
    auc_scores.append(roc_auc)
    plt.plot(fpr, tpr, color=colors[i], lw=2,
             label=f'{CLASS_NAMES[i]} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — DenseNet-121 v2', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

macro_auc = np.mean(auc_scores)
print(f'\nPer-Class AUC:')
for i, cls in enumerate(CLASS_NAMES):
    print(f'  {cls}: {auc_scores[i]:.4f}')
print(f'  Macro AUC: {macro_auc:.4f}')

## Bolum 16: Per-Class Detayli Metrikler

In [ ]:
recall_per_class = recall_score(test_labels, test_preds, average=None)
precision_per_class = precision_score(test_labels, test_preds, average=None)
f1_per_class = f1_score(test_labels, test_preds, average=None)
macro_recall = recall_score(test_labels, test_preds, average='macro')
macro_precision = precision_score(test_labels, test_preds, average='macro')

print('=' * 70)
print('PER-CLASS DETAYLI METRIKLER')
print('=' * 70)
print(f'\n{"Sinif":<8} {"Precision":<12} {"Recall":<12} {"F1-Score":<12} {"Support":<10}')
print('-' * 55)

support = [np.sum(test_labels == i) for i in range(NUM_CLASSES)]
for i, cls in enumerate(CLASS_NAMES):
    print(f'{cls:<8} {precision_per_class[i]:<12.4f} {recall_per_class[i]:<12.4f} {f1_per_class[i]:<12.4f} {support[i]:<10}')

print('-' * 55)
print(f'{"Macro":<8} {macro_precision:<12.4f} {macro_recall:<12.4f} {test_f1:<12.4f} {sum(support):<10}')

print(f'\nTest Accuracy: {test_acc*100:.2f}%')
print(f'Test Loss: {test_loss:.4f}')

## Bolum 17: Baseline & Model Ailesi Karsilastirmasi

In [ ]:
print('=' * 90)
print('v2 vs BASELINE vs EFFICIENTNET-B3 KARSILASTIRMASI')
print('=' * 90)

print(f'\n{"Model":<22} {"Macro F1":<10} {"ACA P":<8} {"ACA R":<8} {"ACA F1":<8} {"MCA P":<8} {"MCA R":<8} {"MCA F1":<8} {"PCA P":<8} {"PCA R":<8} {"PCA F1":<8}')
print('-' * 100)

# Baseline referans
print(f'{"densenet_baseline":<22} {"0.8494":<10} {"0.7541":<8} {"0.8214":<8} {"0.7863":<8} {"0.9710":<8} {"0.8859":<8} {"0.9265":<8} {"0.7486":<8} {"0.9448":<8} {"0.8354":<8}')

# efnb3_v3 referans
print(f'{"efnb3_v3 (best efnb3)":<22} {"0.8479":<10} {"0.7458":<8} {"0.7857":<8} {"0.7652":<8} {"0.9495":<8} {"0.9185":<8} {"0.9337":<8} {"0.8101":<8} {"0.8828":<8} {"0.8449":<8}')

# v2 sonuclari
print(f'{"densenet_v2":<22} {test_f1:<10.4f} {precision_per_class[0]:<8.4f} {recall_per_class[0]:<8.4f} {f1_per_class[0]:<8.4f} {precision_per_class[1]:<8.4f} {recall_per_class[1]:<8.4f} {f1_per_class[1]:<8.4f} {precision_per_class[2]:<8.4f} {recall_per_class[2]:<8.4f} {f1_per_class[2]:<8.4f}')
print('-' * 100)

# Fark analizi
baseline_f1 = 0.8494
diff = test_f1 - baseline_f1
print(f'\n=== v2 vs Baseline ===')
print(f'Macro F1 farki: {diff:+.4f} ({"IYILESME" if diff > 0 else "GERILEME" if diff < 0 else "AYNI"})')
print(f'ACA F1 farki:   {f1_per_class[0] - 0.7863:+.4f}')
print(f'MCA F1 farki:   {f1_per_class[1] - 0.9265:+.4f}')
print(f'PCA F1 farki:   {f1_per_class[2] - 0.8354:+.4f}')

## Bolum 18: Hedef Kontrolu

In [ ]:
print('=' * 70)
print('HEDEF KONTROLU')
print('=' * 70)

targets = {
    'Macro F1 >= 0.87 (BIRINCIL HEDEF)': (test_f1, 0.87),
    'ACA F1 >= 0.80': (f1_per_class[0], 0.80),
    'MCA F1 >= 0.90': (f1_per_class[1], 0.90),
    'PCA F1 >= 0.85': (f1_per_class[2], 0.85),
    'Macro Recall >= 0.83': (macro_recall, 0.83),
}

for target_name, (value, threshold) in targets.items():
    met = value >= threshold
    status = 'ULASILDI' if met else 'ULASILAMADI'
    icon = '+' if met else '-'
    print(f'  [{icon}] {target_name}: {value:.4f} ({status})')

print(f'\n--- Egitim Bilgileri ---')
print(f'Toplam epoch: {len(history["train_loss"])}')
print(f'En iyi Val F1: {best_val_f1:.4f}')
print(f'Final LR: {optimizer.param_groups[0]["lr"]:.2e}')

## Bolum 19: Sonuc Ozeti

Bu bolum, results.md guncellemesi icin gerekli tum metrikleri icerir.

In [ ]:
print('=' * 70)
print('DENSENET-121 v2 — SONUC OZETI')
print('=' * 70)

print(f'\n--- Deney Bilgileri ---')
print(f'Model: DenseNet-121')
print(f'Versiyon: v2')
print(f'Loss: Focal Loss (gamma={GAMMA}, weight_power={WEIGHT_POWER})')
print(f'Softened weights: ACA={class_weights_loss[0]:.4f}, MCA={class_weights_loss[1]:.4f}, PCA={class_weights_loss[2]:.4f}')
print(f'LR: {LEARNING_RATE}')
print(f'Scheduler: CosineAnnealingWarmRestarts (T_0={T_0}, T_mult={T_MULT})')
print(f'IMG_SIZE: {IMG_SIZE}, Batch: {BATCH_SIZE}')
print(f'Early Stopping: patience={PATIENCE}')
print(f'Parametre: {total_params:,}')

print(f'\n--- Egitim ---')
print(f'Toplam epoch: {len(history["train_loss"])}')
print(f'Best Val Macro F1: {best_val_f1:.4f}')

print(f'\n--- Test Sonuclari ---')
print(f'Test Accuracy: {test_acc*100:.2f}%')
print(f'Test Loss: {test_loss:.4f}')

print(f'\n--- Per-Class Metrikler ---')
for i, cls in enumerate(CLASS_NAMES):
    print(f'  {cls}: P={precision_per_class[i]:.4f}, R={recall_per_class[i]:.4f}, F1={f1_per_class[i]:.4f}')

print(f'\n--- Macro Metrikler ---')
print(f'  Macro Precision: {macro_precision:.4f}')
print(f'  Macro Recall:    {macro_recall:.4f}')
print(f'  Macro F1:        {test_f1:.4f}')
print(f'  Macro AUC:       {macro_auc:.4f}')

print(f'\n--- Baseline ile Karsilastirma ---')
print(f'  Baseline Macro F1: 0.8494')
print(f'  v2 Macro F1:       {test_f1:.4f}')
print(f'  Fark:              {test_f1 - 0.8494:+.4f}')

print(f'\n--- results.md icin ---')
print(f'| v2 | {test_acc*100:.2f}% | {test_f1:.4f} | {recall_per_class[0]:.4f} | {recall_per_class[1]:.4f} | {recall_per_class[2]:.4f} | {macro_recall:.4f} | FocalLoss | {LEARNING_RATE} | {GAMMA} | {WEIGHT_POWER} | CosineAnnealing | Focal Loss (gamma={GAMMA}, power={WEIGHT_POWER}). Early stop {{epoch}}. epoch |')

print('\n' + '=' * 70)